# BirdCLEF+ 2026 Perch v2 Probe

Extract **frozen Google Perch v2 embeddings** and train a shallow PyTorch probe for the 206 BirdCLEF+ 2026 primary labels. The notebook tests whether **foundation-model acoustic features** can outperform a lightweight mel-spectrogram CNN and provide teacher signals for later distillation.


## 1. Environment

Configure the Kaggle runtime, attach the **TensorFlow 2.20 wheel input**, and load the libraries used for Perch embedding extraction and PyTorch probe training.


In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)


class CFG:
    seed = 42
    competition_name = "birdclef-2026"
    data_root = None
    artifact_dir = Path("/kaggle/working/artifacts")
    tensorflow_wheel_root = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0")
    perch_input_roots = [
        Path("/kaggle/input/datasets/jaejohn/perch-meta"),
        Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2/2"),
        Path("/kaggle/input/perch-meta"),
    ]


def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)


def find_data_root() -> Path:
    candidates = [
        Path("/kaggle/input/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack/birdclef-2026"),
        Path("/kaggle/input/birdclef-2026-repack"),
    ]
    for path in candidates:
        if (path / "train.csv").exists():
            return path
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = list(input_root.glob("**/train.csv"))
        if matches:
            return matches[0].parent
    raise FileNotFoundError("Could not find train.csv. Attach the BirdCLEF 2026 dataset.")


def package_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


def version_tuple(value: str) -> tuple[int, ...]:
    core = value.split("+")[0].split("-")[0]
    parts = []
    for part in core.split("."):
        if part.isdigit():
            parts.append(int(part))
        else:
            break
    return tuple(parts)


def install_tensorflow_220() -> None:
    tf_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorflow-2.20.0*.whl"))
    tb_wheels = sorted(CFG.tensorflow_wheel_root.glob("**/tensorboard-2.20.0*.whl"))
    if not tf_wheels:
        raise FileNotFoundError(
            f"TensorFlow 2.20 wheel not found under {CFG.tensorflow_wheel_root}. "
            "Attach the bc26-tensorflow-2-20-0 Kaggle input before running notebook 3."
        )
    targets = [str(path) for path in tb_wheels[:1] + tf_wheels[:1]]
    print(f"Installing TensorFlow 2.20 from {CFG.tensorflow_wheel_root}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", *targets], check=True)


seed_everything(CFG.seed)
CFG.data_root = find_data_root()
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

print(f"Data root: {CFG.data_root}")
print(f"Output directory: {CFG.artifact_dir}")

import librosa
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

installed_tf = package_version("tensorflow")
if installed_tf is None or version_tuple(installed_tf) < version_tuple("2.20.0"):
    print(f"TensorFlow {installed_tf} is older than the Perch v2 requirement; installing 2.20.0 from Kaggle input.")
    install_tensorflow_220()

try:
    import tensorflow as tf
except ImportError:
    tf = None

torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)


class CFG(CFG):
    artifact_dir = Path("/kaggle/working/artifacts/perch_v2")
    sample_rate = 32000
    duration = 5.0
    embedding_dim = 1536
    extraction_batch_size = 8
    probe_batch_size = 128
    probe_epochs = 20
    early_stopping_patience = 4
    early_stopping_min_delta = 1e-4
    hidden_dim = 512
    dropout = 0.25
    lr = 1e-3
    max_samples = None
    perch_model_dir = None


CFG.artifact_dir.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if tf is not None:
    print(f"TensorFlow: {tf.__version__}")


## 2. Metadata


Use the same **206 primary labels** as the EfficientNet baseline. Keeping the target space fixed makes the comparison about **representation quality**, not label design.


In [ ]:
train = pd.read_csv(CFG.data_root / "train.csv")
train["filepath"] = train["filename"].map(lambda x: CFG.data_root / "train_audio" / x)
if CFG.max_samples:
    train = train.sample(CFG.max_samples, random_state=CFG.seed).reset_index(drop=True)

labels = sorted(train["primary_label"].unique())
label_to_idx = {label: idx for idx, label in enumerate(labels)}
idx_to_label = {idx: label for label, idx in label_to_idx.items()}
train["target"] = train["primary_label"].map(label_to_idx)
(CFG.artifact_dir / "labels.json").write_text(json.dumps(idx_to_label, indent=2), encoding="utf-8")

print(f"Rows: {len(train):,}")
print(f"Classes: {len(labels):,}")
display(train.head())


## 3. Load Perch v2


Load the **Perch v2 TensorFlow SavedModel** and run a one-batch smoke test. This confirms the input signature, embedding shape, and TensorFlow/XLA compatibility before the full audio pass.


In [ ]:
def find_perch_model_dir() -> Path:
    if CFG.perch_model_dir:
        path = Path(CFG.perch_model_dir)
        if (path / "saved_model.pb").exists():
            return path
        matches = list(path.glob("**/saved_model.pb")) if path.exists() else []
        if matches:
            return matches[0].parent

    search_roots = [path for path in CFG.perch_input_roots if path.exists()]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        search_roots.append(input_root)

    matches = []
    for root in dict.fromkeys(search_roots):
        if (root / "saved_model.pb").exists():
            matches.append(root / "saved_model.pb")
        matches.extend(root.glob("**/saved_model.pb"))
    matches = [path.parent for path in matches if "perch" in str(path).lower() or "vocal" in str(path).lower() or "bird" in str(path).lower()]
    if matches:
        return matches[0]
    raise FileNotFoundError(
        "Could not find a Perch SavedModel. Attach /kaggle/input/datasets/jaejohn/perch-meta, "
        "the Perch/vocalization-classifier Kaggle model, or set CFG.perch_model_dir."
    )


if tf is None:
    raise ImportError("TensorFlow is required for Perch embedding extraction.")

perch_model_dir = find_perch_model_dir()
perch = tf.saved_model.load(str(perch_model_dir))
infer = perch.signatures["serving_default"]
print(f"Perch model: {perch_model_dir}")
print(f"Inputs: {infer.structured_input_signature}")
print(f"Outputs: {infer.structured_outputs}")


def explain_perch_runtime_error(error: Exception) -> None:
    message = str(error)
    if "XlaCallModuleOp with version 10 is not supported" in message:
        raise RuntimeError(
            "The attached Perch SavedModel requires a newer TensorFlow/XLA runtime than this Kaggle image. "
            "The setup cell should install tensorflow>=2.20 before TensorFlow is imported. Restart the "
            "Kaggle session after installation, then run the notebook from the top. If internet is disabled, "
            "attach a Kaggle dataset containing a compatible TensorFlow wheel or use an older Perch SavedModel export."
        ) from error
    raise error


def smoke_test_perch() -> None:
    dummy = tf.zeros((1, int(CFG.sample_rate * CFG.duration)), dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    try:
        if keyword_specs:
            input_name = next(iter(keyword_specs))
            outputs = infer(**{input_name: dummy})
        else:
            outputs = infer(dummy)
    except Exception as error:
        explain_perch_runtime_error(error)
    print({name: tuple(value.shape) for name, value in outputs.items()})


smoke_test_perch()


## 4. Embedding Extraction


Convert each **5-second waveform** into a **1,536-dimensional Perch embedding** and cache the matrix. Once cached, probe experiments can be repeated without rerunning TensorFlow inference.


In [ ]:
def load_audio(path: Path) -> np.ndarray:
    target_len = int(CFG.sample_rate * CFG.duration)
    y, _ = librosa.load(path, sr=CFG.sample_rate, mono=True, duration=CFG.duration)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    return y[:target_len].astype(np.float32)


def run_perch_batch(batch_waveforms: np.ndarray) -> np.ndarray:
    tensor = tf.convert_to_tensor(batch_waveforms, dtype=tf.float32)
    _, keyword_specs = infer.structured_input_signature
    if keyword_specs:
        input_name = next(iter(keyword_specs))
        try:
            outputs = infer(**{input_name: tensor})
        except Exception as error:
            explain_perch_runtime_error(error)
    else:
        try:
            outputs = infer(tensor)
        except Exception as error:
            explain_perch_runtime_error(error)
    arrays = {name: np.asarray(value) for name, value in outputs.items()}
    embedding_name = next(
        (name for name in arrays if "embedding" in name.lower() or "embed" in name.lower()),
        next(iter(arrays)),
    )
    value = arrays[embedding_name]
    if value.ndim == 3:
        value = value.mean(axis=1)
    elif value.ndim > 3:
        value = value.reshape(value.shape[0], -1)
    return value.astype(np.float32)


embeddings_path = CFG.artifact_dir / "train_embeddings.npz"
if embeddings_path.exists():
    saved = np.load(embeddings_path, allow_pickle=True)
    embeddings = saved["embeddings"]
else:
    chunks = []
    waveforms = []
    for path in tqdm(train["filepath"], desc="audio"):
        waveforms.append(load_audio(path))
        if len(waveforms) == CFG.extraction_batch_size:
            chunks.append(run_perch_batch(np.stack(waveforms)))
            waveforms = []
    if waveforms:
        chunks.append(run_perch_batch(np.stack(waveforms)))
    embeddings = np.concatenate(chunks, axis=0)
    np.savez_compressed(
        embeddings_path,
        embeddings=embeddings,
        labels=train["primary_label"].to_numpy(),
        filenames=train["filename"].to_numpy(),
    )

print(f"Embeddings: {embeddings.shape}")
print(f"Saved: {embeddings_path}")


## 5. Probe Model


The probe is deliberately shallow: **LayerNorm -> Linear -> ReLU -> Dropout -> Linear**. Perch should already encode most acoustic structure, so the classifier only needs to learn compact class boundaries on frozen features.


In [ ]:
class PerchProbe(nn.Module):
    def __init__(self, embedding_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(embedding_dim),
            nn.Linear(embedding_dim, CFG.hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(CFG.dropout),
            nn.Linear(CFG.hidden_dim, num_classes),
        )

    def forward(self, x):
        return self.net(x)


x = embeddings.astype(np.float32)
y = train["target"].to_numpy(dtype=np.int64)


def safe_train_valid_split(targets: np.ndarray, test_size: float = 0.2) -> tuple[np.ndarray, np.ndarray]:
    counts = pd.Series(targets).value_counts()
    rare_classes = set(counts[counts < 2].index)
    all_idx = np.arange(len(targets))
    rare_idx = np.array([idx for idx in all_idx if targets[idx] in rare_classes], dtype=np.int64)
    common_idx = np.array([idx for idx in all_idx if targets[idx] not in rare_classes], dtype=np.int64)

    train_common, valid_idx = train_test_split(
        common_idx,
        test_size=test_size,
        random_state=CFG.seed,
        stratify=targets[common_idx],
    )
    train_idx = np.concatenate([train_common, rare_idx])
    return train_idx, valid_idx


train_idx, valid_idx = safe_train_valid_split(y)
valid_meta = train.iloc[valid_idx][["filename", "primary_label", "target"]].reset_index(drop=True)
print(f"Probe train rows: {len(train_idx):,}")
print(f"Probe valid rows: {len(valid_idx):,}")
print(f"Classes with fewer than 2 rows kept in train only: {(pd.Series(y).value_counts() < 2).sum():,}")

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[train_idx]), torch.from_numpy(y[train_idx])),
    batch_size=CFG.probe_batch_size,
    shuffle=True,
)
valid_loader = DataLoader(
    TensorDataset(torch.from_numpy(x[valid_idx]), torch.from_numpy(y[valid_idx])),
    batch_size=CFG.probe_batch_size * 2,
    shuffle=False,
)

model = PerchProbe(embedding_dim=x.shape[1], num_classes=len(labels)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr)


## 6. Training


Rare singleton classes stay in training so the stratified validation split remains valid. The training loop tracks **best validation accuracy**, applies **early stopping**, and writes diagnostic files for per-class analysis.


In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader) -> tuple[float, float]:
    model.eval()
    total_loss = 0.0
    correct = 0
    seen = 0
    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(dim=1) == yb).sum().item()
        seen += xb.size(0)
    return total_loss / max(seen, 1), correct / max(seen, 1)


@torch.no_grad()
def collect_validation_predictions() -> tuple[pd.DataFrame, pd.DataFrame, float]:
    model.eval()
    top1_labels, top1_probs, top5_labels = [], [], []
    top5_hits = []

    for xb, yb in valid_loader:
        xb = xb.to(device)
        probs = torch.softmax(model(xb), dim=1).cpu()
        values, indices = probs.topk(k=min(5, probs.shape[1]), dim=1)
        for row_values, row_indices, target in zip(values, indices, yb):
            labels_top = [idx_to_label[int(idx)] for idx in row_indices]
            top1_labels.append(labels_top[0])
            top1_probs.append(float(row_values[0]))
            top5_labels.append(" ".join(labels_top))
            top5_hits.append(int(int(target) in [int(idx) for idx in row_indices]))

    pred_df = valid_meta.copy()
    pred_df["top1_label"] = top1_labels
    pred_df["top1_prob"] = top1_probs
    pred_df["top5_labels"] = top5_labels
    pred_df["top1_correct"] = pred_df["primary_label"].eq(pred_df["top1_label"]).astype(int)
    pred_df["top5_correct"] = top5_hits

    class_df = (
        pred_df.groupby("primary_label")
        .agg(
            support=("filename", "count"),
            top1_recall=("top1_correct", "mean"),
            top5_recall=("top5_correct", "mean"),
            mean_top1_prob=("top1_prob", "mean"),
        )
        .reset_index()
        .sort_values(["top1_recall", "support"], ascending=[True, False])
    )
    return pred_df, class_df, float(pred_df["top5_correct"].mean())


history = []
best_acc = 0.0
epochs_without_improvement = 0

for epoch in range(1, CFG.probe_epochs + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)

    valid_loss, valid_acc = evaluate(valid_loader)
    row = {
        "epoch": epoch,
        "train_loss": total_loss / max(len(train_loader.dataset), 1),
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
    }
    history.append(row)
    print(row)

    improved = valid_acc > best_acc + CFG.early_stopping_min_delta
    if improved:
        best_acc = valid_acc
        epochs_without_improvement = 0
        torch.save(
            {
                "model": model.state_dict(),
                "label_to_idx": label_to_idx,
                "cfg": {k: v for k, v in CFG.__dict__.items() if not k.startswith("_")},
                "valid_acc": best_acc,
            },
            CFG.artifact_dir / "best_perch_probe.pt",
        )
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CFG.early_stopping_patience:
            print(f"Early stopping after {epoch} epochs; best valid accuracy: {best_acc:.4f}")
            break

history_df = pd.DataFrame(history)
history_df.to_csv(CFG.artifact_dir / "history.csv", index=False)

try:
    best_checkpoint = torch.load(CFG.artifact_dir / "best_perch_probe.pt", map_location=device, weights_only=False)
except TypeError:
    best_checkpoint = torch.load(CFG.artifact_dir / "best_perch_probe.pt", map_location=device)
model.load_state_dict(best_checkpoint["model"])
validation_predictions, per_class_metrics, valid_top5_acc = collect_validation_predictions()
validation_predictions.to_csv(CFG.artifact_dir / "validation_predictions.csv", index=False)
per_class_metrics.to_csv(CFG.artifact_dir / "per_class_validation_metrics.csv", index=False)
summary = {
    "best_valid_acc": float(best_acc),
    "valid_top5_acc": valid_top5_acc,
    "epochs_ran": len(history_df),
    "best_epoch": int(history_df.loc[history_df["valid_acc"].idxmax(), "epoch"]),
    "embedding_shape": list(embeddings.shape),
}
(CFG.artifact_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(f"Best valid accuracy: {best_acc:.4f}")
print(f"Validation top-5 accuracy: {valid_top5_acc:.4f}")
print(f"Saved outputs to {CFG.artifact_dir}")
display(per_class_metrics.head(10))


## 7. Interpretation

- **Perch embeddings** provide the strongest validation signal in this repo and are best used as teacher features or distillation targets.
- **Early stopping** is important: validation peaks quickly while train loss keeps falling, which suggests the probe can overfit frozen features.
- **Per-class recall** and **top-5 hit rate** identify which labels are already well represented by Perch and which labels still need sampling, augmentation, or domain-specific thresholds.
- Direct Perch inference is operationally heavier than EfficientNet, so keep it experimental until hidden-test runtime is proven.


## 8. Artifacts


Package the **embedding cache**, **best probe checkpoint**, **training history**, **validation predictions**, and **per-class metrics** for Kaggle download. Large generated artifacts remain outside Git.


In [ ]:
import zipfile
from IPython.display import FileLink

zip_path = Path("/kaggle/working/birdclef_perch_v2_artifacts.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(CFG.artifact_dir.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(CFG.artifact_dir.parent))

print(f"Wrote {zip_path} ({zip_path.stat().st_size / 1024 / 1024:.2f} MB)")
display(FileLink(zip_path))
